# Edge case testing

This notebook stress-tests rules with sparse and noisy data. Use these cells to verify that rules do not fire below minimum data thresholds and to inspect behavior under label noise.

In [ ]:
# Ensure local imports work when running from the notebooks folder
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..').resolve()))

from datetime import datetime
from src.metrics.baseline import Feeling, MetricResult
from src.rules.index import energy_uplift_after_strength, late_meals_lower_morning_energy, routine_stabilizes_mood
from src.rules.evaluators import evaluate_rules_for_user

print('Imports OK')

## Sparse data: below min points
Verify that rules respect `min_points` and return no insight when data is insufficient.

In [ ]:
# Energy uplift with too few short-term points
short = MetricResult(metric='post_energy_short', window_days=7, value=5.5, data_points=1)
long = MetricResult(metric='post_energy_long', window_days=30, value=3.5, data_points=10)
print('short.data_points=', short.data_points)
res = energy_uplift_after_strength(short, long, threshold=0.8, min_points=3)
print('Result (should be empty):', res)


## Noisy data: noise injection and robustness
Add gaussian noise to simulate measurement error and verify rule stability across small perturbations.

In [ ]:
import random
import statistics

# Base long-term mean and short-term samples
base_long = 3.6
short_samples = [5.0, 5.3, 4.8, 5.1, 5.2]

def noisy_delta(samples, base, noise_sd=0.1, trials=50):
    fired = 0
    for _ in range(trials):
        noisy = [s + random.gauss(0, noise_sd) for s in samples]
        short_metric = MetricResult(metric='post_energy_short', window_days=7, value=statistics.mean(noisy), data_points=len(noisy))
        long_metric = MetricResult(metric='post_energy_long', window_days=30, value=base, data_points=30)
        res = energy_uplift_after_strength(short_metric, long_metric, threshold=0.8, min_points=3)
        if res:
            fired += 1
    return fired

for sd in [0.05, 0.1, 0.2]:
    f = noisy_delta(short_samples, base_long, noise_sd=sd, trials=200)
    print(f'Noise SD={sd:.2f} -> fired {f}/200 times')


## Edge case: zero variance in nonroutine stress
Ensure `routine_stabilizes_mood` handles division-by-zero gracefully.

In [ ]:
# var_non == 0 should return empty dict and not raise
routine = [1.9, 2.0, 2.1]
nonroutine = [2.0, 2.0, 2.0]  # zero variance
res = routine_stabilizes_mood(routine, nonroutine, min_points=3)
print('Result (should be empty):', res)


## Automated test harness (optional)
Add additional cells to run these checks as automated assertions for CI if desired.